# Week 7: Evaluation Framework

Phase 1 마무리 - Pre-Freeze 평가

## 목표
1. **RAGAS 4지표 측정**: Faithfulness, Answer Relevancy, Context Precision, Context Recall
2. **도메인 특화 메트릭**: Refusal Accuracy, Citation Accuracy
3. **RAGAS 한계 사례 분석**: 점수와 실제 품질의 괴리 발굴

## 데이터셋
- Golden Set v1: `data/eval/golden_set_v1.csv` (35 questions, 5 q_types)

In [1]:
import sys
sys.path.insert(0, '..')

import csv
import json
import re
from pathlib import Path
from dataclasses import dataclass
from collections import defaultdict

import pandas as pd
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_core.documents import Document

/Users/joyoungha/Desktop/project/rag-agent-portfolio/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load Golden Set v1

In [2]:
@dataclass
class GoldenQuestion:
    """Golden Set 질문."""
    question: str
    ground_truth: str
    reference_context: str
    q_type: str
    modality_label: str
    notes: str

def load_golden_set(path: Path) -> list[GoldenQuestion]:
    """Load Golden Set v1 from CSV."""
    questions = []
    with open(path, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            questions.append(GoldenQuestion(
                question=row['question'],
                ground_truth=row['ground_truth'],
                reference_context=row['reference_context'],
                q_type=row['q_type'],
                modality_label=row['modality_label'],
                notes=row['notes'],
            ))
    return questions

GOLDEN_SET_PATH = Path('../data/eval/golden_set_v1.csv')
golden_questions = load_golden_set(GOLDEN_SET_PATH)
print(f"Loaded {len(golden_questions)} questions")

# Distribution by q_type
q_type_counts = defaultdict(int)
for q in golden_questions:
    q_type_counts[q.q_type] += 1
print(f"\nBy q_type: {dict(q_type_counts)}")

Loaded 36 questions

By q_type: {'factual': 22, 'multi_hop': 4, 'comparison': 4, 'out_of_scope': 4, 'safety': 2}


## 2. Setup Baseline Retriever (Hybrid + Rerank)

In [3]:
from src.vectorstore import load_vectorstore
from src.retrieval import HybridRerankerRetriever, HybridRetrieverConfig, RerankConfig

CHROMA_DIR = Path('../data/chroma_db_c3')

print("Loading vectorstore...")
vs = load_vectorstore(CHROMA_DIR, collection_name="lg_manuals_c3")
print(f"Loaded {vs._collection.count()} documents")

print("\nCreating Hybrid+Rerank retriever...")
retriever = HybridRerankerRetriever(
    vs,
    hybrid_config=HybridRetrieverConfig(bm25_weight=0.5, dense_weight=0.5),
    rerank_config=RerankConfig(first_stage_k=20, final_k=5),
)
print("Retriever ready.")

Loading vectorstore...


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


Loaded 258 documents

Creating Hybrid+Rerank retriever...


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 6917.78it/s]


Retriever ready.


## 3. RAG Pipeline with Citation

In [4]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

RAG_PROMPT = """다음 컨텍스트를 바탕으로 질문에 답하세요.
반드시 출처(문서명, 페이지)를 명시하세요.
컨텍스트에서 답을 찾을 수 없으면 "제공된 문서에서 확인할 수 없습니다."라고 답하세요.

컨텍스트:
{context}

질문: {question}

답변:"""

def run_rag(question: str, retriever, llm) -> tuple[str, list[Document]]:
    """Run RAG pipeline with citation tracking."""
    docs = retriever.invoke(question)
    
    # Build context with source info
    context_parts = []
    for i, doc in enumerate(docs):
        source = doc.metadata.get('source', 'unknown')
        page = doc.metadata.get('page', '?')
        context_parts.append(f"[출처: {source} p.{page}]\n{doc.page_content}")
    
    context = "\n\n".join(context_parts)
    prompt = RAG_PROMPT.format(context=context, question=question)
    
    response = llm.invoke(prompt)
    return response.content, docs

# Test
test_q = golden_questions[0]
response, docs = run_rag(test_q.question, retriever, llm)
print(f"Q: {test_q.question}")
print(f"A: {response[:200]}...")

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Q: 정수기 필터 교체 주기는 얼마인가요?
A: 정수기 필터 교체 주기는 다음과 같습니다:
- 중금속9 흡착 필터: 6개월 (10 L/일 사용 기준)
- 바이러스 클리어 필터: 12개월 (10 L/일 사용 기준) 
(출처: waterpurifier_simple_WP_KOR_MFL71817002_03_240524_00_OM_WEB.pdf p.1)...


## 4. RAGAS 4지표 측정

In [5]:
# RAGAS 0.4.3+ modern API: instructor LLM + native ragas embeddings
# (The langchain LLM/Embeddings path is deprecated; ragas.metrics.collections
# metrics require InstructorBaseRagasLLM / BaseRagasEmbedding directly.)
from unittest.mock import MagicMock
_fake_vertexai = MagicMock()
_fake_vertexai.ChatVertexAI = MagicMock()
sys.modules["langchain_community.chat_models.vertexai"] = _fake_vertexai

import asyncio
import openai
from ragas.llms import llm_factory
from ragas.embeddings import OpenAIEmbeddings as RagasOpenAIEmbeddings
from ragas.metrics.collections import (
    Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall,
)

# One OpenAI client is reused by both the eval LLM and the eval embeddings.
_openai_client = openai.AsyncOpenAI()  # ragas .ascore() requires an async client
ragas_llm = llm_factory(
    model="gpt-4o-mini", provider="openai", client=_openai_client,
    max_tokens=8192,  # NLI verdict JSON for verbose Korean responses needs headroom
)
ragas_embeddings = RagasOpenAIEmbeddings(
    client=_openai_client, model="text-embedding-3-small",
)


In [6]:
async def evaluate_with_ragas(
    questions: list[GoldenQuestion],
    retriever,
    llm,
    sample_size: int | None = None,
) -> pd.DataFrame:
    """Run RAGAS evaluation on golden set (modern collections API).

    Async because the metric `.ascore()` methods are coroutines and Jupyter
    already owns an event loop. Per-metric exceptions are caught so one bad
    question/metric does not abort the whole batch — failing cells become NaN.
    """
    eval_questions = [q for q in questions if q.q_type != 'out_of_scope']
    if sample_size:
        eval_questions = eval_questions[:sample_size]

    print(f"Evaluating {len(eval_questions)} questions...")

    results = []
    for i, q in enumerate(eval_questions):
        print(f"  [{i+1}/{len(eval_questions)}] {q.question[:30]}...")
        response, docs = run_rag(q.question, retriever, llm)
        results.append({
            'question': q.question,
            'q_type': q.q_type,
            'ground_truth': q.ground_truth,
            'reference_context': q.reference_context,
            'response': response,
            'contexts': [d.page_content for d in docs],
            'docs': docs,
        })

    faithfulness = Faithfulness(llm=ragas_llm)
    answer_relevancy = AnswerRelevancy(llm=ragas_llm, embeddings=ragas_embeddings)
    context_precision = ContextPrecision(llm=ragas_llm)
    context_recall = ContextRecall(llm=ragas_llm)

    def _val(x):
        if isinstance(x, Exception):
            print(f"    metric failed: {type(x).__name__}: {str(x)[:80]}")
            return float('nan')
        return x.value

    async def _score_one(r: dict) -> dict:
        q, resp, ctxs, ref = r['question'], r['response'], r['contexts'], r['ground_truth']
        f, ar, cp, cr = await asyncio.gather(
            faithfulness.ascore(user_input=q, response=resp, retrieved_contexts=ctxs),
            answer_relevancy.ascore(user_input=q, response=resp),
            context_precision.ascore(user_input=q, reference=ref, retrieved_contexts=ctxs),
            context_recall.ascore(user_input=q, retrieved_contexts=ctxs, reference=ref),
            return_exceptions=True,
        )
        return {
            'faithfulness':       _val(f),
            'answer_relevancy':   _val(ar),
            'context_precision':  _val(cp),
            'context_recall':     _val(cr),
        }

    print("\nRunning RAGAS evaluation...")
    scored = await asyncio.gather(*[_score_one(r) for r in results])

    merged = [{**r, **s} for r, s in zip(results, scored)]
    return pd.DataFrame(merged)


In [7]:
# Run RAGAS evaluation (sample for time constraint)
# Set sample_size=None for full evaluation
ragas_results = await evaluate_with_ragas(golden_questions, retriever, llm, sample_size=10)

# Summary statistics
print("\n=== RAGAS 4지표 Summary ===")
print(f"Faithfulness:       {ragas_results['faithfulness'].mean():.3f}")
print(f"Answer Relevancy: {ragas_results['answer_relevancy'].mean():.3f}")
print(f"Context Precision:  {ragas_results['context_precision'].mean():.3f}")
print(f"Context Recall:     {ragas_results['context_recall'].mean():.3f}")

Evaluating 10 questions...
  [1/10] 정수기 필터 교체 주기는 얼마인가요?...
  [2/10] WD523A 모델의 제어창 사용법을 알려주세요...
  [3/10] 물맛이 이상할 때 어떻게 해야 하나요?...
  [4/10] 정수기 출수구 살균 기능은 어떻게 사용하나요?...
  [5/10] 온수 잠금 기능을 설정하는 방법...
  [6/10] AS281DAW 필터 수명은 얼마나 되나요?...
  [7/10] 공기청정기 필터 청소는 어떻게 하나요?...
  [8/10] 공기가 탁할 때 어떤 모드를 사용해야 하나요?...
  [9/10] 공기청정기 센서 청소 방법을 알려주세요...
  [10/10] 상태 표시등이 빨간색일 때 무슨 의미인가요?...

Running RAGAS evaluation...

=== RAGAS 4지표 Summary ===
Faithfulness:       0.842
Answer Relevancy: 0.347
Context Precision:  0.901
Context Recall:     0.700


In [8]:
# By q_type breakdown
print("\n=== By q_type ===")
for q_type in ragas_results['q_type'].unique():
    subset = ragas_results[ragas_results['q_type'] == q_type]
    print(f"\n{q_type} (n={len(subset)}):")
    print(f"  Faithfulness:       {subset['faithfulness'].mean():.3f}")
    print(f"  Answer Relevancy: {subset['answer_relevancy'].mean():.3f}")
    print(f"  Context Precision:  {subset['context_precision'].mean():.3f}")
    print(f"  Context Recall:     {subset['context_recall'].mean():.3f}")


=== By q_type ===

factual (n=10):
  Faithfulness:       0.842
  Answer Relevancy: 0.347
  Context Precision:  0.901
  Context Recall:     0.700


## 5. Refusal Accuracy

In [9]:
REFUSAL_PATTERNS = [
    r"제공된 문서에서 확인할 수 없습니다",
    r"문서에서 확인할 수 없",
    r"정보를 찾을 수 없",
    r"해당 정보가 없",
    r"포함되어 있지 않",
]

def is_refusal(response: str) -> bool:
    """Check if response is a refusal."""
    for pattern in REFUSAL_PATTERNS:
        if re.search(pattern, response):
            return True
    return False

def evaluate_refusal_accuracy(questions: list[GoldenQuestion], retriever, llm) -> dict:
    """Evaluate refusal accuracy.
    
    - out_of_scope questions: should refuse
    - Other questions: should answer
    """
    results = []
    
    for q in questions:
        response, _ = run_rag(q.question, retriever, llm)
        refused = is_refusal(response)
        should_refuse = q.q_type == 'out_of_scope'
        
        correct = (refused == should_refuse)
        
        results.append({
            'question': q.question,
            'q_type': q.q_type,
            'response': response[:100],
            'refused': refused,
            'should_refuse': should_refuse,
            'correct': correct,
        })
    
    # Calculate metrics
    df = pd.DataFrame(results)
    
    # Overall accuracy
    accuracy = df['correct'].mean()
    
    # False positive rate (오거절률): answered questions that were refused
    should_answer = df[~df['should_refuse']]
    fp_rate = should_answer['refused'].mean() if len(should_answer) > 0 else 0
    
    # True positive rate (올바른 거절률): out_of_scope that were refused
    should_refuse_df = df[df['should_refuse']]
    tp_rate = should_refuse_df['refused'].mean() if len(should_refuse_df) > 0 else 0
    
    return {
        'refusal_accuracy': accuracy,
        'false_positive_rate': fp_rate,
        'true_positive_rate': tp_rate,
        'results': df,
    }

In [10]:
# Evaluate refusal accuracy on out_of_scope + safety + sample of others
# out_of_scope: should refuse
# safety: should answer (with safety warning)
# others: should answer

refusal_sample = [
    q for q in golden_questions 
    if q.q_type in ['out_of_scope', 'safety']
] + [q for q in golden_questions if q.q_type == 'factual'][:5]

print(f"Evaluating refusal accuracy on {len(refusal_sample)} questions...")
refusal_result = evaluate_refusal_accuracy(refusal_sample, retriever, llm)

print(f"\n=== Refusal Accuracy ===")
print(f"Overall Accuracy:     {refusal_result['refusal_accuracy']:.1%}")
print(f"True Positive Rate:   {refusal_result['true_positive_rate']:.1%} (out_of_scope correctly refused)")
print(f"False Positive Rate:  {refusal_result['false_positive_rate']:.1%} (answerable incorrectly refused)")

Evaluating refusal accuracy on 11 questions...

=== Refusal Accuracy ===
Overall Accuracy:     100.0%
True Positive Rate:   100.0% (out_of_scope correctly refused)
False Positive Rate:  0.0% (answerable incorrectly refused)


In [11]:
# Show refusal results by q_type
refusal_df = refusal_result['results']
print("\n=== Refusal by q_type ===")
for q_type in refusal_df['q_type'].unique():
    subset = refusal_df[refusal_df['q_type'] == q_type]
    print(f"{q_type}: {subset['correct'].sum()}/{len(subset)} correct")


=== Refusal by q_type ===
out_of_scope: 4/4 correct
safety: 2/2 correct
factual: 5/5 correct


## 6. Citation Accuracy

In [12]:
# Citation patterns: "waterpurifier_simple p.15" or "p.15" etc.
CITATION_PATTERNS = [
    r'(waterpurifier|airpurifier|vacuumcleaner)_\w+\s*p?\.?(\d+)',  # full source with page
    r'p\.?(\d+)',  # just page number
    r'페이지\s*(\d+)',
]

def extract_citations(response: str) -> list[tuple[str, str]]:
    """Extract (source, page) citations from response."""
    citations = []
    
    # Pattern 1: full source name with page
    matches = re.findall(r'((?:waterpurifier|airpurifier|vacuumcleaner)_\w+)\s*p?\.?(\d+)', response, re.IGNORECASE)
    for source, page in matches:
        citations.append((source.lower(), page))
    
    # Pattern 2: just page (assume from context)
    if not citations:
        page_matches = re.findall(r'p\.?(\d+)', response)
        for page in page_matches:
            citations.append(('unknown', page))
    
    return citations

def check_citation_accuracy(response: str, reference_context: str, docs: list[Document]) -> dict:
    """Check if citations in response match reference and retrieved docs."""
    if reference_context == 'N/A':
        # out_of_scope - should have no citation
        citations = extract_citations(response)
        return {
            'has_citation': len(citations) > 0,
            'should_cite': False,
            'correct': len(citations) == 0,
            'reason': 'out_of_scope - no citation expected',
        }
    
    # Parse reference context
    ref_match = re.search(r'(\w+)\s*p?\.?(\d+)', reference_context)
    if not ref_match:
        return {
            'has_citation': False,
            'should_cite': True,
            'correct': False,
            'reason': 'could not parse reference',
        }
    
    ref_source = ref_match.group(1).lower()
    ref_page = int(ref_match.group(2))
    
    # Extract citations from response
    citations = extract_citations(response)
    
    if not citations:
        return {
            'has_citation': False,
            'should_cite': True,
            'correct': False,
            'reason': 'no citation in response',
        }
    
    # Check if any citation matches (with ±2 page tolerance)
    for cite_source, cite_page in citations:
        cite_page = int(cite_page)
        # Source match (partial) and page within ±2
        source_match = ref_source in cite_source or cite_source in ref_source or cite_source == 'unknown'
        page_match = abs(cite_page - ref_page) <= 2
        
        if source_match and page_match:
            return {
                'has_citation': True,
                'should_cite': True,
                'correct': True,
                'reason': f'matched {cite_source} p.{cite_page} to ref {ref_source} p.{ref_page}',
            }
    
    return {
        'has_citation': True,
        'should_cite': True,
        'correct': False,
        'reason': f'citations {citations} did not match ref {ref_source} p.{ref_page}',
    }

def evaluate_citation_accuracy(questions: list[GoldenQuestion], retriever, llm) -> dict:
    """Evaluate citation accuracy."""
    results = []
    
    for q in questions:
        response, docs = run_rag(q.question, retriever, llm)
        check = check_citation_accuracy(response, q.reference_context, docs)
        
        results.append({
            'question': q.question,
            'q_type': q.q_type,
            'reference_context': q.reference_context,
            'response': response[:150],
            **check,
        })
    
    df = pd.DataFrame(results)
    
    # Only count questions that should have citations
    should_cite = df[df['should_cite']]
    accuracy = should_cite['correct'].mean() if len(should_cite) > 0 else 0
    
    return {
        'citation_accuracy': accuracy,
        'total_should_cite': len(should_cite),
        'correct_citations': should_cite['correct'].sum(),
        'results': df,
    }

In [13]:
# Evaluate citation accuracy on sample
citation_sample = [q for q in golden_questions if q.q_type in ['factual', 'comparison', 'multi_hop', 'safety']][:10]

print(f"Evaluating citation accuracy on {len(citation_sample)} questions...")
citation_result = evaluate_citation_accuracy(citation_sample, retriever, llm)

print(f"\n=== Citation Accuracy ===")
print(f"Accuracy: {citation_result['citation_accuracy']:.1%}")
print(f"Correct: {citation_result['correct_citations']}/{citation_result['total_should_cite']}")

Evaluating citation accuracy on 10 questions...

=== Citation Accuracy ===
Accuracy: 0.0%
Correct: 0/10


## 7. RAGAS 한계 사례 분석

In [14]:
# Find cases where RAGAS score and actual quality diverge

def analyze_ragas_limits(ragas_df: pd.DataFrame) -> dict:
    """Find RAGAS limitation cases."""
    limits = {
        'high_score_bad_answer': [],  # RAGAS high but actually bad
        'low_score_good_answer': [],  # RAGAS low but actually good
    }
    
    for _, row in ragas_df.iterrows():
        faith = row['faithfulness']
        relevancy = row['answer_relevancy']
        
        # High score but potentially bad answer
        # Look for cases where faithfulness is high but answer is short/incomplete
        response_len = len(row['response'])
        if faith > 0.8 and response_len < 50:
            limits['high_score_bad_answer'].append({
                'question': row['question'],
                'response': row['response'],
                'faithfulness': faith,
                'reason': 'High faithfulness but very short response - may miss details',
            })
        
        # Low score but potentially good answer
        if faith < 0.5 and response_len > 100:
            limits['low_score_good_answer'].append({
                'question': row['question'],
                'response': row['response'],
                'faithfulness': faith,
                'reason': 'Low faithfulness but detailed response - may be over-penalized',
            })
    
    return limits

In [15]:
# Analyze RAGAS limits from the evaluation results
if 'ragas_results' in dir() and len(ragas_results) > 0:
    limits = analyze_ragas_limits(ragas_results)
    
    print("=== RAGAS 한계 사례 ===")
    print(f"\n점수 높은데 답변 나쁜 사례: {len(limits['high_score_bad_answer'])}건")
    for case in limits['high_score_bad_answer'][:2]:
        print(f"  Q: {case['question'][:50]}...")
        print(f"  A: {case['response'][:80]}...")
        print(f"  Faithfulness: {case['faithfulness']:.2f}")
        print(f"  분석: {case['reason']}")
        print()
    
    print(f"\n점수 낮은데 답변 괜찮은 사례: {len(limits['low_score_good_answer'])}건")
    for case in limits['low_score_good_answer'][:2]:
        print(f"  Q: {case['question'][:50]}...")
        print(f"  A: {case['response'][:80]}...")
        print(f"  Faithfulness: {case['faithfulness']:.2f}")
        print(f"  분석: {case['reason']}")
        print()
else:
    print("RAGAS 결과가 없습니다. 먼저 섹션 4를 실행하세요.")

=== RAGAS 한계 사례 ===

점수 높은데 답변 나쁜 사례: 0건

점수 낮은데 답변 괜찮은 사례: 0건


## 8. Save Results

In [16]:
# Save evaluation results
RESULTS_PATH = Path('../data/week7_evaluation_results.json')

results_summary = {
    'golden_set_size': len(golden_questions),
    'ragas': {
        'faithfulness': float(ragas_results['faithfulness'].mean()) if 'ragas_results' in dir() else None,
        'answer_relevancy': float(ragas_results['answer_relevancy'].mean()) if 'ragas_results' in dir() else None,
        'context_precision': float(ragas_results['context_precision'].mean()) if 'ragas_results' in dir() else None,
        'context_recall': float(ragas_results['context_recall'].mean()) if 'ragas_results' in dir() else None,
    },
    'refusal_accuracy': refusal_result['refusal_accuracy'] if 'refusal_result' in dir() else None,
    'citation_accuracy': citation_result['citation_accuracy'] if 'citation_result' in dir() else None,
}

with open(RESULTS_PATH, 'w', encoding='utf-8') as f:
    json.dump(results_summary, f, indent=2, ensure_ascii=False)

print(f"Results saved to {RESULTS_PATH}")
print(json.dumps(results_summary, indent=2, ensure_ascii=False))

Results saved to ../data/week7_evaluation_results.json
{
  "golden_set_size": 36,
  "ragas": {
    "faithfulness": 0.8416666666666666,
    "answer_relevancy": 0.34741830520930195,
    "context_precision": 0.9013888888682245,
    "context_recall": 0.7
  },
  "refusal_accuracy": 1.0,
  "citation_accuracy": 0.0
}


## Summary

### 1. 최종 점수 (sample_size=10, all `factual`)

| Metric | Score | 판단 |
|---|---|---|
| Faithfulness | **0.842** | ✅ 임계 ≥0.8 통과. 응답이 컨텍스트에 충실. |
| Answer Relevancy | **0.347** | ⚠️ 임계 한참 미달. **점수 자체보다 측정 신뢰성을 의심** (아래 §4 참고). |
| Context Precision | **0.901** | ✅ 임계 통과. 검색된 청크가 reference에 잘 기여. |
| Context Recall | **0.700** | ⚠️ 임계 ≥0.8 미달. reference 일부가 검색 컨텍스트로 직접 뒷받침 안 됨. |
| Refusal Accuracy | **1.000** | ✅ out_of_scope 4/4 + safety 2/2 + factual 5/5 모두 정답. |
| Citation Accuracy | **0.000** | ❌ **cell 18 regex 버그** — 실제 인용 품질이 아님 (아래 §4 참고). |

> Raw JSON: `data/week7_evaluation_results.json` (cell 24 출력).

---

### 2. RAGAS 4지표 정리

| 지표 | 무엇을 보는가 | 계산 방식 | 실용 임계 | 알려진 한계 |
|---|---|---|---|---|
| **Faithfulness** | 응답 ↔ 컨텍스트 | 응답을 LLM이 statement 단위로 분해 → 각 statement가 컨텍스트로 NLI 추론 가능한지 판정 → 지지된 비율 | ≥ 0.8 | (1) 짧은 응답은 statement 수가 적어 통계가 불안정. (2) 의역·요약을 NLI가 종종 over-penalize. |
| **Answer Relevancy** | 응답 ↔ 질문 | 응답으로부터 LLM이 N=3개의 가상 질문 생성 → 임베딩 → 원래 user_input과 평균 cosine 유사도 | ≥ 0.7 | (1) 거절 응답은 의도와 무관하게 점수 낮음. **(2) 한국어 임베딩 품질에 직접 의존 — 이번 결과의 0.347 원인 추정.** |
| **Context Precision** | 컨텍스트 ↔ reference | 검색된 청크들이 reference 도출에 기여하는지 LLM이 청크별로 판정 → MAP 형태 가중 평균 | ≥ 0.8 | (1) reference가 짧으면 다수 청크가 "무관"으로 판정되어 보수적. (2) 중복 청크가 후순위면 0 처리되어 평균 하락. |
| **Context Recall** | reference ↔ 컨텍스트 | reference 텍스트를 statement로 분해 → 각 statement가 검색된 컨텍스트로 뒷받침되는지 비율 | ≥ 0.8 | (1) reference에 추론·요약 진술이 섞이면 직접 뒷받침이 어려워 점수 낮음. (2) reference 자체 품질·세분성에 강하게 의존. |

---

### 3. 도메인 특화 지표 정리

| 지표 | 정의 | 실용 임계 | 측정 의의 |
|---|---|---|---|
| **Refusal Accuracy** | `out_of_scope` 질문을 거절(REFUSAL_PATTERNS 매칭)하고 answerable 질문에는 답하는 binary correctness 평균 | ≥ 0.9 | hallucination 차단 능력. **이번에 100% — 매우 강건.** |
| **Citation Accuracy** | 응답에 추출된 (source, page) 인용이 reference_context와 ±2 페이지 이내 일치하는 비율. `should_cite=True`만 분모. | ≥ 0.7 | 사용자가 답변의 근거를 직접 검증 가능한지. 가전 매뉴얼 도메인에서 특히 중요. |

---

### 4. 점수와 실제 품질의 괴리 — Phase 1 종합 분석

이번 run에서 cell 22의 자동 휴리스틱(`analyze_ragas_limits`)은 0건을 잡았지만, **육안으로 보면 두 가지 명백한 괴리가 있음:**

#### 4-1. Answer Relevancy 0.347은 모델 품질이 아니라 측정 한계

- Faithfulness 0.842, Context Precision 0.901 → 응답이 컨텍스트로 잘 뒷받침되고, 검색도 정확.
- 그런데 Answer Relevancy만 0.347 → **불일치는 측정 도구 쪽 문제일 가능성이 큼**.
- 가설: 한국어 응답 → 한국어 가상 질문 생성 → `text-embedding-3-small`로 임베딩 → 원 질문과 cosine. `text-embedding-3-small`은 한국어에 최적화돼 있지 않아 동일 의미 문장 간 cosine이 낮게 나오는 경향.
- **검증 액션 (next week):** 동일 데이터셋에서 임베딩만 `text-embedding-3-large` 또는 다국어 모델(`bge-m3`)로 바꿔 같은 metric 재측정 → 점수 상승 폭으로 측정 신뢰성 평가.

#### 4-2. Citation Accuracy 0%는 모델이 아니라 평가 코드 버그

- 응답은 실제로 출처를 인용함 (cell 7 샘플: `[출처: waterpurifier_simple_WP_KOR_MFL71817002_03_240524_00_OM_WEB.pdf p.1]`).
- cell 18의 정규식 `((?:waterpurifier|airpurifier|vacuumcleaner)_\w+)\s*p?\.?(\d+)`은 `_\w+` 다음에 곧장 페이지를 기대하는데, 실제로는 `.pdf p.1`이 중간에 들어가 매칭 실패.
- **검증 액션:** regex를 `((?:waterpurifier|airpurifier|vacuumcleaner)_\w+)[^\d]*?p\.?\s*(\d+)` 정도로 수정 후 재측정 → 실제 인용 정확도 산출.

#### 4-3. 이번 sample이 전부 `factual` (sample_size=10 우연)

- 골든셋 36문항 중 factual이 22문항이라 앞 10개가 다 factual로 잡힘.
- **multi_hop, comparison, safety는 평가되지 않음.** Phase 1 종결 판단을 하려면 stratified sampling으로 q_type별 최소 N개 평가 필요.

---

### 5. Phase 1 → Phase 2 시사점

- **텍스트 RAG는 factual에서 baseline 통과**: Faithfulness/Precision은 임계 위, Refusal 100%.
- **Context Recall 0.700**은 reference 분해 시 일부 statement가 retrieved chunk로 직접 뒷받침되지 않는다는 신호 → Phase 2 cross-modal 도입으로 그림 기반 절차 진술을 채우면 개선 여지.
- **RAGAS raw score만으로 판단 금지** 케이스 확인:
  - 거절 응답 → Answer Relevancy 점수 낮음 (의도 무시)
  - 한국어 응답 → Answer Relevancy 자체가 임베딩 품질에 끌려 내려감 (이번 결과)
  - reference가 여러 페이지에 분산 → Context Recall 분해가 어려워 보수적 점수
- **Phase 2 시작 전 권장사항** (ADR-007 평가 신뢰성):
  1. Golden Set v2: q_type별 stratified, reference_context 페이지 단위 세분화
  2. 임베딩 모델 한국어 검증 (text-embedding-3-large vs bge-m3 비교)
  3. Citation regex 수정 + 실제 인용 정확도 재측정
